In [60]:
import pandas as pd

You’re working on an ads‑analytics pipeline. Product managers want to compute campaign‑level performance metrics, but the raw data is messy and spread across multiple event tables. Your task will be to clean, join, dedupe, and compute metrics.

To simulate this, here are three small datasets (20–30 rows total):

campaigns — static metadata

impressions — raw ad impression logs

clicks — click events

spend — daily spend logs (intentionally imperfect)

All datasets are small enough to paste directly into pandas.

In [59]:
%reset -f

In [61]:
#Dataset 1 — campaigns (8 rows)

campaigns = pd.DataFrame([
    [101, "Shoes Launch", "2024-01-01", "2024-01-31"],
    [102, "Winter Jackets", "2024-01-10", "2024-02-15"],
    [103, "Socks Promo", "2024-01-05", "2024-01-20"],
    [104, "Sportswear", "2024-01-01", "2024-03-01"],
    [105, "Clearance Sale", "2024-01-15", "2024-01-25"],
    [106, "New Arrivals", "2024-01-20", "2024-02-10"],
    [107, "Accessories", "2024-01-01", "2024-01-15"],
    [108, "Gift Cards", "2024-01-10", "2024-01-31"],
], columns=["campaign_id", "campaign_name", "start_date", "end_date"])


In [62]:
#📘 Dataset 2 — impressions (20 rows)
#Includes duplicates, out‑of‑range timestamps, and missing campaign IDs.

impressions = pd.DataFrame([
    [1, 101, "2024-01-01 09:00", "US"],
    [2, 101, "2024-01-01 09:05", "US"],
    [3, 102, "2024-01-10 12:00", "CA"],
    [4, 102, "2024-01-10 12:00", "CA"],   # duplicate timestamp
    [5, 103, "2024-01-06 14:00", "US"],
    [6, 103, "2024-01-25 14:00", "US"],   # outside campaign window
    [7, 104, "2024-01-02 08:00", "UK"],
    [8, 104, "2024-01-02 08:00", "UK"],   # duplicate row
    [9, 105, "2024-01-15 10:00", "US"],
    [10, 105, "2024-01-16 10:00", "US"],
    [11, 106, "2024-01-20 11:00", "CA"],
    [12, 106, "2024-01-21 11:00", "CA"],
    [13, 107, "2024-01-01 09:30", "US"],
    [14, 107, "2024-01-16 09:30", "US"],  # outside window
    [15, 108, "2024-01-10 13:00", "US"],
    [16, 108, "2024-01-11 13:00", "US"],
    [17, None, "2024-01-12 13:00", "US"], # missing campaign_id
    [18, 102, "2024-02-20 12:00", "CA"],  # outside window
    [19, 104, "2024-01-03 08:00", "UK"],
    [20, 104, "2024-01-03 08:00", "UK"],  # duplicate
], columns=["impression_id", "campaign_id", "timestamp", "country"])


In [63]:
# Dataset 3 — clicks (10 rows)
#Includes multiple clicks per impression and some impressions that never clicked.

clicks = pd.DataFrame([
    [1, 1, "2024-01-01 09:01"],
    [2, 2, "2024-01-01 09:06"],
    [3, 3, "2024-01-10 12:01"],
    [4, 5, "2024-01-06 14:05"],
    [5, 7, "2024-01-02 08:05"],
    [6, 7, "2024-01-02 08:06"],  # multiple clicks
    [7, 9, "2024-01-15 10:05"],
    [8, 11, "2024-01-20 11:05"],
    [9, 15, "2024-01-10 13:05"],
    [10, 20, "2024-01-03 08:05"],
], columns=["click_id", "impression_id", "timestamp"])


In [64]:
# Dataset 4 — spend (12 rows)
#Includes missing days, extra days, and a duplicated entry.

spend = pd.DataFrame([
    [101, "2024-01-01", 120],
    [101, "2024-01-02", 130],
    [102, "2024-01-10", 200],
    [102, "2024-01-11", 210],
    [103, "2024-01-06", 50],
    [104, "2024-01-02", 300],
    [104, "2024-01-02", 300],  # duplicate
    [105, "2024-01-15", 80],
    [106, "2024-01-20", 150],
    [107, "2024-01-01", 60],
    [108, "2024-01-10", 40],
    [108, "2024-01-12", 45],
], columns=["campaign_id", "date", "spend"])




Using pandas, produce a campaign‑level performance table with:

impressions (deduped, valid date range)

clicks (unique clicks per impression)

CTR

CTR
= clicks/impressions

total spend

CPC

CPC
= spend/clicks

country breakdown (% of impressions by country)

flag campaigns with invalid impressions (outside campaign window or missing campaign_id)

This is intentionally realistic:
You’ll need joins, deduping, date filtering, grouping, and some window logic.

In [ ]:
# Drop rows with missing campaign_id (in place)
impressions.dropna(subset=['campaign_id'], inplace=True)

# Drop duplicate impression rows (in place)
cols_to_check = impressions.columns.difference(['impression_id'])
impressions.drop_duplicates(subset=cols_to_check, inplace=True)

# Convert timestamp to datetime (in place)
impressions['timestamp'] = pd.to_datetime(impressions['timestamp'])

# Convert campaign dates (in place)
campaigns['start_date'] = pd.to_datetime(campaigns['start_date'])
campaigns['end_date'] = pd.to_datetime(campaigns['end_date'])

# Merge only the needed columns (overwriting the same df)
impressions = impressions.merge(
    campaigns[['campaign_id', 'start_date', 'end_date']].rename(columns={'start_date':'campaign_start_date',  'end_date':'campaign_end_date'}),
    on='campaign_id',
    how='left'
)

# Filter to valid date window (boolean mask, no extra df)
mask = (
    (impressions['timestamp'] >= impressions['campaign_start_date']) &
    (impressions['timestamp'] <= impressions['campaign_end_date'])
)

impressions_clean = impressions[mask]

# Compute impressions per campaign
impressions_per_campaign = (
    impressions_clean.groupby('campaign_id')['impression_id']
    .count()
    .reset_index(name='total_impressions')
)


In [58]:
impressions[mask]

/tmp/ipykernel_3527/781597877.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  impressions[mask]


,impression_id,campaign_id,timestamp,country,campaign_start_date,campaign_end_date
0,1,101.0,2024-01-01 09:00:00,US,2024-01-01,2024-01-31
1,2,101.0,2024-01-01 09:05:00,US,2024-01-01,2024-01-31
2,3,102.0,2024-01-10 12:00:00,CA,2024-01-10,2024-02-15
3,5,103.0,2024-01-06 14:00:00,US,2024-01-05,2024-01-20
5,7,104.0,2024-01-02 08:00:00,UK,2024-01-01,2024-03-01
6,9,105.0,2024-01-15 10:00:00,US,2024-01-15,2024-01-25
7,10,105.0,2024-01-16 10:00:00,US,2024-01-15,2024-01-25
8,11,106.0,2024-01-20 11:00:00,CA,2024-01-20,2024-02-10
9,12,106.0,2024-01-21 11:00:00,CA,2024-01-20,2024-02-10
10,13,107.0,2024-01-01 09:30:00,US,2024-01-01,2024-01-15


In [57]:
impressions_per_campaign

,campaign_id,total_impressions
0,101.0,2
1,102.0,1
2,103.0,1
3,104.0,2
4,105.0,2
5,106.0,2
6,107.0,1
7,108.0,2


In [65]:
#Q2. Compute total unique clicks per campaign.